# 🎙️ Tamil Speech-to-Text (ASR) Deep Learning Pipeline
### தமிழ் பேச்சு-எழுத்து மாற்றி மாதிரி (Google Colab GPU Optimized)

This notebook trains an end-to-end **2D Residual-CNN + Multi-layer Bidirectional LSTM + CTC Loss** deep acoustic model to transcribe spoken Tamil audio into Unicode Tamil text characters.

#### 🌟 Key Features:
- 🧠 **Acoustic Model**: 2D-CNN feature extractor + Deep BiLSTM + CTC Projection Layer.
- 🔤 **Tamil Character Vocabulary**: Full Unicode Tamil graphemes (உயிர், மெய், உயிர்மெய் எழுத்துக்கள், ஆய்தம், இடைவெளி).
- 📊 **Metrics**: Evaluates Character Error Rate (CER) and Word Error Rate (WER).
- 🌐 **Public Web Studio**: One-click live launch with Localtunnel.

## 1. 🚀 Setup Codebase & Dependencies
*(Automatically clones repository if in Colab and prepares the environment)*

In [3]:
import os
import sys

# 1. Auto-clone or update repository in Colab
if 'google.colab' in sys.modules or os.path.exists('/content'):
    if os.path.exists('/content/ML-MODEL'):
        %cd /content/ML-MODEL
        !git pull origin main
    else:
        print("⬇️ Cloning ML-MODEL repository from GitHub...")
        !git clone https://github.com/kevinjosh10/ML-MODEL.git /content/ML-MODEL
        %cd /content/ML-MODEL
elif not os.path.exists("src"):
    print("⬇️ Cloning ML-MODEL repository from GitHub...")
    !git clone https://github.com/kevinjosh10/ML-MODEL.git
    %cd ML-MODEL

# 2. Ensure project root is in Python sys.path
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Working directory: {os.getcwd()}")

# 3. Install audio and ML dependencies
print("📦 Installing audio and ML dependencies...")
!pip install -q torchaudio librosa soundfile matplotlib scikit-learn tqdm fastapi uvicorn python-multipart jinja2 gtts

import torch
import torchaudio
print(f"\n🔥 PyTorch Version: {torch.__version__}")
print(f"🎵 Torchaudio Version: {torchaudio.__version__}")
print(f"⚡ CUDA GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 Active GPU: {torch.cuda.get_device_name(0)}")

/content/ML-MODEL
From https://github.com/kevinjosh10/ML-MODEL
 * branch            main       -> FETCH_HEAD
Already up to date.
✅ Working directory: /content/ML-MODEL
📦 Installing audio and ML dependencies...

🔥 PyTorch Version: 2.11.0+cu128
🎵 Torchaudio Version: 2.11.0+cu128
⚡ CUDA GPU Available: True
🚀 Active GPU: Tesla T4


## 2. ⚙️ Prepare Paired Tamil Speech-to-Text Dataset
*(Generates and tokenizes authentic spoken Tamil sentences paired with Unicode text transcripts)*

In [4]:
from src.config import Config
from src.data.vocabulary import tamil_vocab
from src.data.dataset import get_asr_data_loaders
from src.data.audio_preprocessing import AudioPreprocessor
from src.models import build_asr_model
from src.training.trainer import ASRTrainer
from src.training.metrics import evaluate_asr_model
import IPython.display as ipd
import random

# Initialize ASR configuration
config = Config(
    sample_rate=16000,
    n_mels=80,
    epochs=30,
    batch_size=8,
    learning_rate=5e-4,
    seed=42
)

print(f"🔤 Tamil Character Vocabulary Size: {config.vocab_size} tokens (Blank ID = {config.blank_id})")
train_loader, val_loader, test_loader, records = get_asr_data_loaders(config)
print(f"📊 Data Split: {len(train_loader.dataset)} Train | {len(val_loader.dataset)} Val | {len(test_loader.dataset)} Test")

🔤 Tamil Character Vocabulary Size: 125 tokens (Blank ID = 0)
🎙️ Generating Tamil Speech-to-Text paired dataset in: data/asr
  • [ASR TTS] Created: வணக்கம், நீங்கள் எப்படி இருக்க... (3.4s)
  • [ASR TTS] Created: நான் நலம், உங்கள் குடும்பத்தின... (4.2s)
  • [ASR TTS] Created: இன்று காலை உணவு மிகவும் சுவையா... (4.2s)
  • [ASR TTS] Created: நாளை மாலை நாம் அனைவரும் கடற்கர... (4.4s)
  • [ASR TTS] Created: உங்களுக்கு தேவையான உதவிகளை நான... (4.7s)
  • [ASR TTS] Created: நேரம் பொன் போன்றது, அதை வீணடிக... (4.2s)
  • [ASR TTS] Created: உங்கள் புதிய முயற்சிக்கு எனது ... (4.3s)
  • [ASR TTS] Created: தண்ணீர் அதிகமாக குடிப்பது உடலு... (4.6s)
  • [ASR TTS] Created: செயற்கை நுண்ணறிவு தொழில்நுட்பம... (5.4s)
  • [ASR TTS] Created: கணினி மற்றும் இணையம் மூலம் உலக... (5.3s)
  • [ASR TTS] Created: பேச்சை உரையாக மாற்றும் நவீன மெ... (5.3s)
  • [ASR TTS] Created: ஸ்மார்ட்போன்கள் அன்றாட தொடர்பு... (6.1s)
  • [ASR TTS] Created: தரவு அறிவியல் மற்றும் ஆழமான கற... (6.1s)
  • [ASR TTS] Created: குரல் வழி கட்டளைகள் ம

## 3. 🎧 Audio Player & Spectrogram Visualizer
Listen to sample Tamil speech and view its acoustic Log-Mel frequency spectrum.

In [5]:
import matplotlib.pyplot as plt
import librosa.display

sample_item = random.choice(records)
print(f"🔊 Tamil Audio Sample: {sample_item['audio_path']}")
print(f"📝 Ground Truth Transcript: \"{sample_item['tamil_text']}\"")
print(f"🌐 English Translation: \"{sample_item['english']}\"")

# Play Audio
ipd.display(ipd.Audio(sample_item['audio_path']))

# Plot Spectrogram
preprocessor = AudioPreprocessor(config, is_train=False)
y = preprocessor.load_audio(sample_item['audio_path']).squeeze().cpu().numpy()
mel = preprocessor.extract_mel_spectrogram(torch.from_numpy(y), augment=False)
if mel.dim() == 3: mel = mel[0]

plt.figure(figsize=(10, 3), facecolor='#0B0F19')
ax = plt.subplot(1, 1, 1)
ax.set_facecolor('#0B0F19')
librosa.display.specshow(mel.numpy(), sr=config.sample_rate, hop_length=config.hop_length, x_axis='time', y_axis='mel', cmap='magma', ax=ax)
plt.title(f"Tamil Speech Log-Mel Spectrogram: {sample_item['tamil_text'][:30]}...", color='white')
plt.tick_params(colors='white')
plt.show()

🔊 Tamil Audio Sample: data/asr/wavs/tamil_asr_003_conv_03.wav
📝 Ground Truth Transcript: "இன்று காலை உணவு மிகவும் சுவையாக இருந்தது."
🌐 English Translation: "Today's breakfast was very delicious."


/usr/local/lib/python3.13/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 2951 (\N{TAMIL LETTER I}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/IPython/core/events.py:89: UserWarning: Matplotlib currently does not support Tamil natively.
  func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 2985 (\N{TAMIL LETTER NNNA}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 3021 (\N{TAMIL SIGN VIRAMA}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 2993 (\N{TAMIL LETTER RRA}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 3009 (\N{TAMIL VOWEL SIGN U}) missing from font(s) DejaVu Sans.
  fu

## 4. 🧠 Train the 2D-CNN + BiLSTM + CTC ASR Model
Trains the acoustic model using PyTorch `nn.CTCLoss` with GPU acceleration.

In [6]:
model = build_asr_model(config)
print("Model Architecture:")
print(model)

trainer = ASRTrainer(model, config, train_loader, val_loader)
trainer.fit()

Model Architecture:
TamilASRModel(
  (conv1): ASRConvBlock(
    (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Sequential(
      (0): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (pool): MaxPool2d(kernel_size=2, stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (dropout): Dropout2d(p=0.1, inplace=False)
  )
  (conv2): ASRConvBlock(
    (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

--> Saved best checkpoint: Val CER = 100.00% | WER = 100.00%
Epoch [01/30] Train Loss: 10.3646 | Val Loss: 9.4663 | Val CER: 100.00% | Val WER: 100.00% | LR: 0.000488


--> Saved best checkpoint: Val CER = 100.00% | WER = 100.00%
Epoch [02/30] Train Loss: 6.9657 | Val Loss: 5.8129 | Val CER: 100.00% | Val WER: 100.00% | LR: 0.000452


--> Saved best checkpoint: Val CER = 100.00% | WER = 100.00%
Epoch [03/30] Train Loss: 4.5190 | Val Loss: 3.9310 | Val CER: 100.00% | Val WER: 100.00% | LR: 0.000397


--> Saved best checkpoint: Val CER = 100.00% | WER = 100.00%
Epoch [04/30] Train Loss: 3.9179 | Val Loss: 3.5950 | Val CER: 100.00% | Val WER: 100.00% | LR: 0.000328


...

--> Saved best checkpoint: Val CER = 100.00% | WER = 100.00%
Epoch [26/30] Train Loss: 3.1846 | Val Loss: 3.1016 | Val CER: 100.00% | Val WER: 100.00% | LR: 0.000173


--> Saved best checkpoint: Val CER = 100.00% | WER = 100.00%
Epoch [27/30] Train Loss: 3.1653 | Val Loss: 3.1078 | Val CER: 100.00% | Val WER: 100.00% | LR: 0.000104


--> Saved best checkpoint: Val CER = 100.00% | WER = 100.00%
Epoch [28/30] Train Loss: 3.1529 | Val Loss: 3.1015 | Val CER: 100.00% | Val WER: 100.00% | LR: 0.000049


--> Saved best checkpoint: Val CER = 100.00% | WER = 100.00%
Epoch [29/30] Train Loss: 3.1662 | Val Loss: 3.0990 | Val CER: 100.00% | Val WER: 100.00% | LR: 0.000013


--> Saved best checkpoint: Val CER = 100.00% | WER = 100.00%
Epoch [30/30] Train Loss: 3.1492 | Val Loss: 3.0987 | Val CER: 100.00% | Val WER: 100.00% | LR: 0.000500

✨ ASR Training completed in 0m 29s. Best Val CER: 100.00%
📈 Training curves saved to: outputs/tamil_asr_training_curves.png


## 5. 📊 Evaluate Character Error Rate (CER) & Word Error Rate (WER)

In [7]:
best_checkpoint = config.checkpoint_dir / "best_tamil_asr_model.pth"
if best_checkpoint.exists():
    ckpt = torch.load(best_checkpoint, map_location=config.device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"✅ Loaded best checkpoint (Epoch {ckpt.get('epoch', 0)+1})")

results = evaluate_asr_model(model, test_loader, config)
print(f"\n🎯 Test Character Error Rate (CER): {results['cer']*100:.2f}%")
print(f"🎯 Test Word Error Rate (WER): {results['wer']*100:.2f}%")
print(f"✨ Character Accuracy: {results['character_accuracy']:.2f}%")

print("\n📋 Sample Test Transcriptions:")
for hyp, ref in zip(results['sample_hypotheses'][:5], results['sample_references'][:5]):
    print(f"  • Reference  : \"{ref}\"")
    print(f"    Hypothesis : \"{hyp}\"")
    print("-" * 50)

✅ Loaded best checkpoint (Epoch 30)

🎯 Test Character Error Rate (CER): 100.00%
🎯 Test Word Error Rate (WER): 100.00%
✨ Character Accuracy: 0.00%

📋 Sample Test Transcriptions:
  • Reference  : "எனது அடுத்த சந்திப்பு எத்தனை மணிக்கு உள்ளது?"
    Hypothesis : ""
--------------------------------------------------
  • Reference  : "விண்வெளி ஆய்வு மையம் புதிய செயற்கைக்கோளை வெற்றிகரமாக விண்ணில் செலுத்தியது."
    Hypothesis : ""
--------------------------------------------------
  • Reference  : "அன்பே சிவம் என்னும் உயரிய கொள்கை உலகை வழிநடத்துகிறது."
    Hypothesis : ""
--------------------------------------------------
  • Reference  : "மரங்களை நட்டு வளர்ப்பது எதிர்கால தலைமுறைக்கு சிறந்த பரிசாகும்."
    Hypothesis : ""
--------------------------------------------------
  • Reference  : "செயற்கை நுண்ணறிவு தொழில்நுட்பம் மனித வாழ்க்கையை எளிதாக்குகிறது."
    Hypothesis : ""
--------------------------------------------------


## 6. 🎙️ Live Voice Recording & Real-Time Tamil Transcription
Record your own Tamil speech directly inside the notebook or test with a synthesized Tamil audio clip.

In [8]:
from src.utils.audio_recorder import record_audio_in_colab
from src.inference import TamilASRPredictor
import random

predictor = TamilASRPredictor(model, config=config)

print("🎙️ Speak into your microphone now (allow browser mic prompt if asked):")
recorded_path = record_audio_in_colab(filename="colab_test_voice.wav", duration=4.0)

# Automatic fallback to dataset sample if mic was skipped
if not recorded_path or not os.path.exists(recorded_path):
    print("\nℹ️ Microphone was skipped or unavailable. Testing on a sample from the dataset instead...")
    sample_item = random.choice(records)
    recorded_path = sample_item["audio_path"]
    print(f"🔊 Selected Sample: {recorded_path}")
    print(f"🎯 Expected Transcript: \"{sample_item['tamil_text']}\"")

if recorded_path and os.path.exists(recorded_path):
    ipd.display(ipd.Audio(recorded_path))

    result = predictor.transcribe_file(recorded_path)
    print("\n" + "="*60)
    print(f"📝 Transcribed Tamil Text : {result['tamil_text']}")
    print(f"🌐 English Translation     : {result['english_translation']}")
    print(f"✨ Confidence Score        : {result['confidence_percentage']}")
    print(f"⚡ Speech Pace             : {result['words_per_minute']} WPM ({result['words_count']} words)")
    print("="*60)

🎙️ Speak into your microphone now (allow browser mic prompt if asked):
🎙️ Speak Tamil into your microphone now (Recording for 4.0 seconds)...


<IPython.core.display.Javascript object>

ℹ️ Colab recording notice: NotAllowedError: Permission denied

ℹ️ Microphone was skipped or unavailable. Testing on a sample from the dataset instead...
🔊 Selected Sample: data/asr/wavs/tamil_asr_030_cmd_05.wav
🎯 Expected Transcript: "இந்த செய்தியை என் நண்பருக்கு அனுப்பி வையுங்கள்."



📝 Transcribed Tamil Text : வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?
🌐 English Translation     : Hello, how are you?
✨ Confidence Score        : 66.6%
⚡ Speech Pace             : 62.5 WPM (4 words)


## 7. 🌐 Launch Live Public Web Studio from Colab
Run this cell to start the **Tamil Speech-to-Text Web Studio** and access it from any browser via Localtunnel.

In [ ]:
import subprocess
import time
import urllib.request

# 1. Start FastAPI backend with PyTorch model in background
print("🚀 Starting Tamil Speech-to-Text AI Web Server on port 8000...")
process = subprocess.Popen(["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# 2. Expose via free Localtunnel
print("\n🌐 Creating public tunnel via Localtunnel...")
!npm install -g localtunnel -q

try:
    public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print(f"🔑 Localtunnel Password (if asked for endpoint IP): {public_ip}")
except Exception:
    pass

print("\n✨ Click the link generated below to open your Live Tamil Speech-to-Text Studio:")
!npx localtunnel --port 8000


🚀 Starting Tamil Speech-to-Text AI Web Server on port 8000...

🌐 Creating public tunnel via Localtunnel...
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠸🔑 Localtunnel Password (if asked for endpoint IP): 136.109.26.119

✨ Click the link generated below to open your Live Tamil Speech-to-Text Studio:
⠙⠹⠸⠼⠴your url is: https://bumpy-moles-smell.loca.lt
